In [2]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("C:\\Users\\saqli\\Downloads\\workspace\\Employee-Attrition-Project\\data\\employee_attrition_clean.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (1000, 8)


,EmployeeID,TotalMonthsOfExp,TotalOrgsWorked,MonthsInOrg,LastPayIncrementBand,AverageFeedback,LastPromotionYears,Attrition
0,1,110,4,9,5,4,4,1
1,2,103,3,51,1,4,2,0
2,3,41,4,16,5,4,4,1
3,4,32,4,17,5,2,3,0
4,5,80,3,16,3,4,2,0


In [3]:
X = df.drop(columns=["EmployeeID", "Attrition"])
y = df["Attrition"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

Feature shape: (1000, 6)
Target shape: (1000,)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (800, 6)
X_test: (200, 6)
y_train: (800,)
y_test: (200,)


In [5]:
model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

In [6]:
model.fit(X_train, y_train)

print("Model training completed!")

Model training completed!


In [7]:
y_pred = model.predict(X_test)

print("Actual labels:")
print(y_test.head(10).tolist())

print("Predicted labels:")
print(y_pred[:10].tolist())

Actual labels:
[0, 0, 0, 0, 0, 0, 1, 0, 0, 0]
Predicted labels:
[0, 0, 0, 0, 0, 0, 1, 0, 0, 0]


In [8]:
accuracy = accuracy_score(y_test, y_pred)

print("Test Accuracy:", round(accuracy * 100, 2), "%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Test Accuracy: 100.0 %

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       154
           1       1.00      1.00      1.00        46

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



In [9]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[154   0]
 [  0  46]]


In [10]:
# Check duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Check whether the target is accidentally included in X
print("Feature columns:", X.columns.tolist())

# Check target distribution
print("Target distribution:")
print(y.value_counts())

Duplicate rows: 0
Feature columns: ['TotalMonthsOfExp', 'TotalOrgsWorked', 'MonthsInOrg', 'LastPayIncrementBand', 'AverageFeedback', 'LastPromotionYears']
Target distribution:
Attrition
0    770
1    230
Name: count, dtype: int64


In [11]:
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

print("Training Accuracy:",
      round(accuracy_score(y_train, train_pred) * 100, 2), "%")

print("Testing Accuracy:",
      round(accuracy_score(y_test, test_pred) * 100, 2), "%")

Training Accuracy: 100.0 %
Testing Accuracy: 100.0 %


In [12]:
# Check whether any single feature perfectly separates Attrition classes

for col in X.columns:
    print(f"\nFeature: {col}")
    print(pd.crosstab(df[col], df["Attrition"]))


Feature: TotalMonthsOfExp
Attrition          0  1
TotalMonthsOfExp       
1                  5  2
2                  5  1
3                  9  3
4                  2  1
5                  8  1
...               .. ..
116                7  2
117               12  1
118                5  1
119                4  2
120                4  3

[120 rows x 2 columns]

Feature: TotalOrgsWorked
Attrition          0   1
TotalOrgsWorked         
1                145  42
2                140  48
3                167  41
4                168  48
5                150  51

Feature: MonthsInOrg
Attrition     0  1
MonthsInOrg       
0            10  0
1            25  5
2            31  5
3            25  6
4            21  9
...          .. ..
108           2  1
112           0  1
113           1  0
115           1  0
116           2  0

[110 rows x 2 columns]

Feature: LastPayIncrementBand
Attrition               0   1
LastPayIncrementBand         
1                     163  28
2                     

In [13]:
# Find duplicate feature combinations, excluding EmployeeID and Attrition
feature_cols = X.columns.tolist()

check_df = df[feature_cols + ["Attrition"]]

print("Duplicate feature combinations:",
      check_df.duplicated(subset=feature_cols).sum())

print("\nTarget values for repeated feature combinations:")
print(
    check_df.groupby(feature_cols)["Attrition"]
    .nunique()
    .value_counts()
)

Duplicate feature combinations: 0

Target values for repeated feature combinations:
Attrition
1    1000
Name: count, dtype: int64


In [14]:
print("Original target distribution:")
print(df["Attrition"].value_counts(dropna=False))

print("\nTarget distribution in y:")
print(y.value_counts(dropna=False))

print("\nTraining target distribution:")
print(y_train.value_counts(dropna=False))

print("\nTesting target distribution:")
print(y_test.value_counts(dropna=False))

print("\nFeature columns:")
print(X.columns.tolist())

Original target distribution:
Attrition
0    770
1    230
Name: count, dtype: int64

Target distribution in y:
Attrition
0    770
1    230
Name: count, dtype: int64

Training target distribution:
Attrition
0    616
1    184
Name: count, dtype: int64

Testing target distribution:
Attrition
0    154
1     46
Name: count, dtype: int64

Feature columns:
['TotalMonthsOfExp', 'TotalOrgsWorked', 'MonthsInOrg', 'LastPayIncrementBand', 'AverageFeedback', 'LastPromotionYears']


In [15]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

scores = cross_val_score(
    cv_model, X, y,
    cv=cv,
    scoring="accuracy"
)

print("Fold accuracies:", scores)
print("Mean CV accuracy:", round(scores.mean() * 100, 2), "%")

Fold accuracies: [1. 1. 1. 1. 1.]
Mean CV accuracy: 100.0 %


In [16]:
print(pd.crosstab(df["LastPromotionYears"], df["Attrition"]))

Attrition             0    1
LastPromotionYears          
1                   255    0
2                   266    0
3                   249    0
4                     0  230


In [17]:
X_without_promotion = df.drop(
    columns=["EmployeeID", "Attrition", "LastPromotionYears"]
)
y = df["Attrition"]

X_train, X_test, y_train, y_test = train_test_split(
    X_without_promotion,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model_without_promotion = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

model_without_promotion.fit(X_train, y_train)
y_pred_without_promotion = model_without_promotion.predict(X_test)

print(
    "Test Accuracy:",
    round(accuracy_score(y_test, y_pred_without_promotion) * 100, 2),
    "%"
)
print(classification_report(
    y_test, y_pred_without_promotion, zero_division=0
))

Test Accuracy: 75.5 %
              precision    recall  f1-score   support

           0       0.77      0.96      0.86       154
           1       0.33      0.07      0.11        46

    accuracy                           0.76       200
   macro avg       0.55      0.51      0.48       200
weighted avg       0.67      0.76      0.69       200



In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    classification_report,
    balanced_accuracy_score
)

models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=5, class_weight="balanced", random_state=42
    ),
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced"
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42
    )
}

for name, clf in models.items():
    clf.fit(X_train, y_train)
    predictions = clf.predict(X_test)

    print(f"\n{'=' * 45}")
    print(name)
    print("Balanced accuracy:",
          round(balanced_accuracy_score(y_test, predictions), 3))
    print(classification_report(
        y_test, predictions, zero_division=0
    ))


Decision Tree
Balanced accuracy: 0.455
              precision    recall  f1-score   support

           0       0.72      0.28      0.40       154
           1       0.21      0.63      0.31        46

    accuracy                           0.36       200
   macro avg       0.46      0.45      0.36       200
weighted avg       0.60      0.36      0.38       200


Logistic Regression
Balanced accuracy: 0.536
              precision    recall  f1-score   support

           0       0.80      0.51      0.62       154
           1       0.25      0.57      0.35        46

    accuracy                           0.52       200
   macro avg       0.53      0.54      0.49       200
weighted avg       0.67      0.52      0.56       200


Random Forest
Balanced accuracy: 0.473
              precision    recall  f1-score   support

           0       0.76      0.77      0.77       154
           1       0.19      0.17      0.18        46

    accuracy                           0.64       200
  

The initial Decision Tree model achieved 100% accuracy on both the training and testing sets. Further investigation identified a perfect relationship between LastPromotionYears and the target variable in the dataset. To investigate this relationship, an additional experiment was conducted without that feature. The resulting models showed lower performance, indicating that the original result requires further validation before it can be interpreted as evidence of generalizable predictive performance.